# London House Price Pipeline — Bug Analysis and Correction
## Prompt B — Run 1

This notebook analyses a buggy machine learning pipeline for predicting London house prices,
identifies all bugs, explains their impact, and provides a corrected implementation.


## Original (Buggy) Pipeline

```python
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load data
prices = pd.read_csv('london_house_prices.csv')
areas = pd.read_csv('london_area_features.csv')
df = prices.merge(areas, on='outcode', how='left')

# Prepare features
feature_cols = ['bedrooms', 'bathrooms', 'floorAreaSqM', 'livingRooms',
                'rentEstimate', 'crime_total', 'census_denom_total']
X = df[feature_cols].fillna(0)
y = df['price']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate
train_preds = model.predict(X_train)
rmse = np.sqrt(mean_squared_error(y_train, train_preds))
mae = mean_absolute_error(y_train, train_preds)
r2 = r2_score(y_train, train_preds)

print(f"Test RMSE: £{rmse:,.0f}")
print(f"Test MAE:  £{mae:,.0f}")
print(f"Test R²:   {r2:.4f}")
```


---
## Bug Analysis

Below is a detailed analysis of every bug found in the pipeline.


### Bug 1: Target Leakage — `rentEstimate` included as a feature

**Location:** `feature_cols` list (line containing `'rentEstimate'`)

**Problem:** `rentEstimate` is a rent estimate for the property, which is directly derived from property market values. It has an extremely high correlation with the sale price (~0.98). Including it as a predictor constitutes **target leakage** — the model is effectively using a proxy of the target variable as an input feature.

**Impact on results:** This artificially inflates all performance metrics. The model appears to perform brilliantly (R² ≈ 0.999) because it's essentially "cheating" — it has access to information that encodes the answer. In production, a new property would not have a reliable rent estimate before its price is known, making this feature unavailable at prediction time.

**Fix:** Remove `rentEstimate` from the feature list and use additional legitimate features instead.

---

### Bug 2: Evaluating on TRAINING data but reporting as "Test" metrics

**Location:** Evaluate section — `model.predict(X_train)` and `mean_squared_error(y_train, train_preds)`

**Problem:** The code generates predictions on `X_train` (training data) and evaluates against `y_train`, but prints the results labelled as "Test RMSE", "Test MAE", and "Test R²". This is a critical error: **the model is being evaluated on the same data it was trained on**, not on the held-out test set.

**Impact on results:** Training metrics are always overly optimistic, especially for flexible models like Random Forest which can memorise training data. The reported R² of 0.9988 is a training R², not a genuine test R². The true test performance would be significantly worse, giving a false sense of model quality.

**Fix:** Use `X_test` and `y_test` for evaluation: `model.predict(X_test)` and evaluate against `y_test`.

---

### Bug 3: No log-transform on the skewed target variable

**Location:** Target variable preparation (`y = df['price']`)

**Problem:** London house prices are **heavily right-skewed** (skewness > 5). Training a model directly on raw prices means the model is dominated by extreme values, and errors on expensive properties disproportionately affect the loss function.

**Impact on results:** Without log-transformation, the model's RMSE is inflated by extreme outliers, and the model struggles to accurately predict both cheap and expensive properties. A log-transform would produce more normally distributed errors and better overall predictions.

**Fix:** Apply `np.log1p()` to the target before training and `np.expm1()` to predictions for evaluation.

---

### Bug 4: `fillna(0)` is inappropriate for missing value imputation

**Location:** `X = df[feature_cols].fillna(0)`

**Problem:** Filling missing values with 0 is semantically incorrect for most features. For example, a missing `floorAreaSqM` of 0 implies the property has zero floor area, which is nonsensical. Similarly, 0 bedrooms or 0 bathrooms creates artificial data points that don't represent reality.

**Impact on results:** This introduces noise and bias. Properties with missing data are treated as having zero-valued features, pulling model predictions in incorrect directions. The effect is especially pronounced for `floorAreaSqM`, where 0 is far from the true median.

**Fix:** Use median imputation for numeric features, which preserves the distribution and is more statistically appropriate.

---

### Bug 5: No outlier handling for extreme prices

**Location:** No outlier treatment anywhere in the pipeline

**Problem:** The dataset contains extreme outliers in the price column (multi-million pound properties). These extreme values have a disproportionate effect on model training, especially affecting RMSE which squares the errors.

**Impact on results:** The model wastes capacity trying to fit extreme values, reducing accuracy on the majority of properties. RMSE is particularly inflated by these outliers.

**Fix:** Cap prices at the 99th percentile or use a robust outlier removal strategy before training.

---

### Bug 6: Very limited feature set

**Location:** `feature_cols` list — only 7 features used

**Problem:** The pipeline only uses 7 features from the merged dataset, ignoring many informative area-level features (crime breakdowns, census demographics, POI counts) and important property features (propertyType, tenure, energyRating, latitude, longitude).

**Impact on results:** The model's predictive power is unnecessarily limited. Including property type, location, and richer area-level features would substantially improve performance.

**Fix:** Include all relevant features, encode categoricals, and use the full merged dataset.


---
## Running the Buggy Pipeline (for comparison)

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

# ===== BUGGY PIPELINE (original) =====
prices = pd.read_csv('london_house_prices.csv')
areas = pd.read_csv('london_area_features.csv')
df = prices.merge(areas, on='outcode', how='left')

feature_cols_buggy = ['bedrooms', 'bathrooms', 'floorAreaSqM', 'livingRooms',
                      'rentEstimate', 'crime_total', 'census_denom_total']
X_buggy = df[feature_cols_buggy].fillna(0)
y_buggy = df['price']

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_buggy, y_buggy, test_size=0.2, random_state=42
)

model_buggy = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model_buggy.fit(X_train_b, y_train_b)

# Bug: evaluating on TRAINING set
train_preds = model_buggy.predict(X_train_b)
buggy_rmse = np.sqrt(mean_squared_error(y_train_b, train_preds))
buggy_mae = mean_absolute_error(y_train_b, train_preds)
buggy_r2 = r2_score(y_train_b, train_preds)

print("BUGGY PIPELINE RESULTS (as reported — actually TRAINING metrics):")
print(f"  'Test' RMSE: \u00a3{buggy_rmse:,.0f}")
print(f"  'Test' MAE:  \u00a3{buggy_mae:,.0f}")
print(f"  'Test' R\u00b2:   {buggy_r2:.4f}")

# What the actual TEST metrics would be (still with leakage)
test_preds_buggy = model_buggy.predict(X_test_b)
actual_test_rmse = np.sqrt(mean_squared_error(y_test_b, test_preds_buggy))
actual_test_r2 = r2_score(y_test_b, test_preds_buggy)
print(f"\nActual TEST metrics (still with rentEstimate leakage):")
print(f"  RMSE: \u00a3{actual_test_rmse:,.0f}")
print(f"  R\u00b2:   {actual_test_r2:.4f}")
print("\nEven the true test R\u00b2 is inflated due to rentEstimate leakage.")


BUGGY PIPELINE RESULTS (as reported — actually TRAINING metrics):
  'Test' RMSE: £31,870
  'Test' MAE:  £6,460
  'Test' R²:   0.9988



Actual TEST metrics (still with rentEstimate leakage):
  RMSE: £65,460
  R²:   0.9950

Even the true test R² is inflated due to rentEstimate leakage.


---
## Corrected Pipeline

The following corrected pipeline addresses ALL identified bugs:
1. Removes `rentEstimate` (target leakage)
2. Evaluates on the **test set** (not training set)
3. Applies **log-transform** to the target
4. Uses **median imputation** instead of fillna(0)
5. **Caps outliers** at the 99th percentile
6. Uses a **comprehensive feature set** including categoricals


In [2]:
# ===== CORRECTED PIPELINE =====
print("="*60)
print("CORRECTED PIPELINE")
print("="*60)

# 1. Load and merge data (same as original — this part was correct)
prices = pd.read_csv('london_house_prices.csv')
areas = pd.read_csv('london_area_features.csv')
df = prices.merge(areas, on='outcode', how='left')
print(f"Merged dataset: {df.shape}")

# 2. FIX: Exclude rentEstimate (target leakage) and identifiers
exclude_cols = ['price', 'rentEstimate', 'outcode', 'outcode_lat', 'outcode_lon']
feature_cols = [c for c in df.columns if c not in exclude_cols]

cat_cols = df[feature_cols].select_dtypes(include='object').columns.tolist()
num_cols = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
print(f"Features: {len(feature_cols)} ({len(num_cols)} numeric, {len(cat_cols)} categorical)")
print(f"Categorical: {cat_cols}")
print(f"rentEstimate EXCLUDED (target leakage)")

# 3. FIX: Median imputation (not fillna(0))
df_clean = df.copy()
for col in num_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())
for col in cat_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

# 4. FIX: Remove rows with missing price
df_clean = df_clean.dropna(subset=['price'])

# 5. FIX: Cap outliers at 99th percentile
p99 = df_clean['price'].quantile(0.99)
n_before = len(df_clean)
df_clean = df_clean[df_clean['price'] <= p99]
print(f"Outlier capping: removed {n_before - len(df_clean):,} rows above \u00a3{p99:,.0f}")

# 6. FIX: Log-transform the target
df_clean['log_price'] = np.log1p(df_clean['price'])

all_features = num_cols + cat_cols
X = df_clean[all_features]
y_log = df_clean['log_price']
y_actual = df_clean['price']

print(f"\nFinal dataset: {X.shape}")


CORRECTED PIPELINE


Merged dataset: (417561, 63)


Features: 58 (55 numeric, 3 categorical)
Categorical: ['propertyType', 'tenure', 'energyRating']
rentEstimate EXCLUDED (target leakage)


Outlier capping: removed 4,173 rows above £4,959,000

Final dataset: (413388, 58)


In [3]:
# Train/test split (same parameters as original)
X_train, X_test, y_train_log, y_test_log = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)
y_test_actual = df_clean.loc[y_test_log.index, 'price']

print(f"Train: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}")


Train: 330,710  |  Test: 82,678


In [4]:
# Train model (same as original — RandomForest with n_estimators=100)
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train_log)

# 7. FIX: Evaluate on TEST set (not training set)
test_preds_log = model.predict(X_test)
test_preds = np.expm1(test_preds_log)  # Back-transform from log space

corrected_rmse = np.sqrt(mean_squared_error(y_test_actual, test_preds))
corrected_mae = mean_absolute_error(y_test_actual, test_preds)
corrected_r2 = r2_score(y_test_actual, test_preds)

print("CORRECTED PIPELINE — Test Set Metrics:")
print(f"  RMSE: \u00a3{corrected_rmse:,.0f}")
print(f"  MAE:  \u00a3{corrected_mae:,.0f}")
print(f"  R\u00b2:   {corrected_r2:.4f}")


CORRECTED PIPELINE — Test Set Metrics:
  RMSE: £119,633
  MAE:  £36,751
  R²:   0.9692


In [5]:
print("="*70)
print("BUGGY vs CORRECTED COMPARISON")
print("="*70)

print(f"\n{'Metric':<15} {'Buggy (reported)':<25} {'Corrected (true test)':<25}")
print("-"*65)
print(f"{'RMSE':<15} {'\u00a3'+f'{buggy_rmse:,.0f}':<25} {'\u00a3'+f'{corrected_rmse:,.0f}':<25}")
print(f"{'MAE':<15} {'\u00a3'+f'{buggy_mae:,.0f}':<25} {'\u00a3'+f'{corrected_mae:,.0f}':<25}")
print(f"{'R\u00b2':<15} {f'{buggy_r2:.4f}':<25} {f'{corrected_r2:.4f}':<25}")

print(f"\nKey differences:")
print(f"  - Buggy R\u00b2 of {buggy_r2:.4f} was inflated by:")
print(f"    (a) evaluating on training data (overfitting)")
print(f"    (b) including rentEstimate (target leakage)")
print(f"  - Corrected R\u00b2 of {corrected_r2:.4f} is a realistic estimate of")
print(f"    true out-of-sample performance.")


BUGGY vs CORRECTED COMPARISON

Metric          Buggy (reported)          Corrected (true test)    
-----------------------------------------------------------------
RMSE            £31,870                   £119,633                 
MAE             £6,460                    £36,751                  
R²              0.9988                    0.9692                   

Key differences:
  - Buggy R² of 0.9988 was inflated by:
    (a) evaluating on training data (overfitting)
    (b) including rentEstimate (target leakage)
  - Corrected R² of 0.9692 is a realistic estimate of
    true out-of-sample performance.


In [6]:
# Predicted vs actual for corrected model
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(y_test_actual, test_preds, alpha=0.1, s=5, c='steelblue')
cap = y_test_actual.quantile(0.99)
ax.plot([0, cap], [0, cap], 'r--', lw=1.5, label='Perfect prediction')
ax.set_xlim(0, cap); ax.set_ylim(0, cap)
ax.set_xlabel('Actual Price (\u00a3)')
ax.set_ylabel('Predicted Price (\u00a3)')
ax.set_title(f'Corrected Pipeline: Predicted vs Actual (R\u00b2={corrected_r2:.4f})')
ax.legend()
plt.tight_layout()
plt.savefig('run3_corrected_pred_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()


In [7]:
# Feature importances
imp = pd.Series(model.feature_importances_, index=all_features).sort_values(ascending=False)
print("Top 15 Feature Importances (Corrected Model):")
for i, (f, v) in enumerate(imp.head(15).items()):
    print(f"  {i+1:2d}. {f:<35s} {v:.4f}")

fig, ax = plt.subplots(figsize=(10, 8))
imp.head(20).plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Importance')
ax.set_title('Feature Importances — Corrected Model')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('run3_feature_importances.png', dpi=150, bbox_inches='tight')
plt.show()


Top 15 Feature Importances (Corrected Model):
   1. floorAreaSqM                        0.5330
   2. census_level4_perc                  0.1119
   3. crime_other_crime                   0.0797
   4. longitude                           0.0525
   5. latitude                            0.0448
   6. census_employed_total_perc          0.0283
   7. census_unemployed_perc              0.0210
   8. tenure                              0.0202
   9. bedrooms                            0.0129
  10. census_age_16_to_34_perc            0.0129
  11. propertyType                        0.0124
  12. crime_criminal_damage_and_arson     0.0086
  13. energyRating                        0.0079
  14. bathrooms                           0.0066
  15. crime_shoplifting                   0.0051


---
## Summary of Bugs and Their Effects

| # | Bug | Impact | Fix |
|---|-----|--------|-----|
| 1 | **rentEstimate included** (target leakage) | Massively inflated R² by providing a near-perfect proxy of the target | Removed from feature set |
| 2 | **Training evaluation labelled as test** | Reported overly optimistic metrics (RF memorises training data) | Evaluate on `X_test`, `y_test` |
| 3 | **No log-transform** on skewed prices | Model dominated by expensive outliers; poor fit across price range | Applied `np.log1p()` to target |
| 4 | **fillna(0)** for missing values | Introduced nonsensical values (0 bedrooms, 0 floor area) | Used median imputation |
| 5 | **No outlier handling** | Extreme prices distorted training and inflated RMSE | Capped at 99th percentile |
| 6 | **Limited feature set** (7 features) | Ignored property type, location, many area features | Used full feature set with encoding |

The buggy pipeline reported a misleadingly high R² ≈ 0.999 due to the combination of training-set evaluation and target leakage via rentEstimate. The corrected pipeline provides honest, generalisable metrics on the held-out test set.
